# Evidencia de aprendizaje 3 — Optimización de procesos de desarrollo

**Programación para el Análisis de Datos** · Institución Universitaria Digital de Antioquia

**Integrantes:** Estefanía Martínez Guzmán · Carlos Andrés Osorio Agudelo · Valentina Pérez Flórez

Este cuaderno ejecuta el proyecto completo que está en el repositorio: extrae el catálogo
de tenis deportivos para mujer de Calzatodo, limpia los datos, calcula los indicadores,
genera el tablero web y comprueba que las pruebas automáticas pasan. Es el mismo código
que corre en GitHub Actions; aquí se ve paso a paso y con las salidas a la vista.

Para ejecutarlo: `Entorno de ejecución > Ejecutar todas`. Requiere las dependencias de
`requirements.txt` y conexión a internet (solo la extracción sale a la red).

## 1. El problema, con un caso concreto

Calzatodo publica 20 publicaciones por página en la categoría de tenis deportivos para
mujer. Comparar precios a mano significa abrir cada ficha, anotar referencia, marca y
precio, y repetir la tarea cada semana. Con dos páginas son 40 publicaciones; en un
catálogo completo serían cientos.

La pregunta que guía el proyecto es simple: **¿qué marcas y qué rangos de precio
dominan la categoría, y cómo mantener esa respuesta actualizada sin trabajo manual?**

La respuesta se construye en cuatro pasos, que son las secciones siguientes: extraer
(scraper), limpiar y validar (reglas de calidad), resumir (métricas) y publicar
(tablero). Al final, GitHub Actions repite esos pasos solo.

In [1]:
from pathlib import Path
import os, sys

# El cuaderno vive en notebooks/; la raíz del repositorio es su carpeta padre.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from ea3.config import load_settings
from ea3 import scraper, transform, report

print("Raíz del proyecto:", RAIZ)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
ajustes = load_settings()
print("Categoría:", ajustes.base_url)
print("Páginas configuradas:", ajustes.pages)

Raíz del proyecto: /home/andresosoa/Documentos/Universidad/2026_2/bloque_2/Programacion_analisis_de_datos/EA3/repo
Python: 3.13.5 | pandas: 3.1.1
Categoría: https://www.calzatodo.com.co/es/mujeres/tenis/deportivo
Páginas configuradas: 2


## 2. Extracción: leer los metadatos que publica el sitio

La página no se raspa por clases CSS. Calzatodo publica un bloque
`<script type="application/ld+json">` con un objeto `ItemList`: una lista ordenada de
productos con su marca, referencia (`sku`) y precios (`lowPrice` / `highPrice`).
Ese formato se llama **JSON-LD** y es un estándar de datos estructurados para que los
buscadores entiendan la página; para nosotros es una fuente estable y legible.

Un ejemplo real de un elemento (recortado):

```json
{
  "@type": "ListItem",
  "position": 1,
  "item": {
    "@type": "Product",
    "name": "Tenis Deportivo de Mujer Skechers 117863wlv",
    "sku": "461841",
    "brand": {"name": "Skechers"},
    "offers": {"lowPrice": 239920, "highPrice": 239920, "priceCurrency": "COP"}
  }
}
```

El scraper descarga la página, busca ese bloque, ignora los demás (`BreadcrumbList`) y
convierte cada elemento en un objeto `Producto`.

In [2]:
sesion = scraper._sesion(ajustes)
productos_pagina = []
for intento in range(3):
    html = scraper.descargar_pagina(sesion, ajustes, pagina=1)
    productos_pagina = scraper.parsear_item_list(html, categoria=ajustes.base_url)
    if productos_pagina:
        break
    print(f"Intento {intento + 1}: la página no trajo el bloque ItemList, se reintenta")

assert productos_pagina, "El sitio no publicó el bloque ItemList en esta ejecución"
print(f"Publicaciones leídas en la página 1: {len(productos_pagina)}")
pd.DataFrame([p.a_fila() for p in productos_pagina]).head(5)

Publicaciones leídas en la página 1: 20


,Nombre,Marca,SKU de referencia,Precio mínimo,Precio máximo,Moneda,Disponible,Categoría,URL,Fecha de extracción
0,Tenis Deportivo de Mujer Skechers 117863wlv,Skechers,461841,239920.0,239920.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117863wlv-2...,2026-11-26T16:00:16-05:00
1,Tenis Deportivo de Mujer Skechers 117704bbk,Skechers,461799,199920.0,199920.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117704bbk-2...,2026-11-26T16:00:16-05:00
2,Tenis Deportivo de Mujer Skechers 117679ofwt,Skechers,461787,229900.0,229900.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117679ofwt-...,2026-11-26T16:00:16-05:00
3,Tenis Deportivo de Mujer Skechers 117861ltmv,Skechers,461835,249900.0,249900.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117861ltmv-...,2026-11-26T16:00:16-05:00
4,Tenis Deportivo de Mujer Skechers 117736wht,Skechers,461817,223920.0,223920.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117736wht-2...,2026-11-26T16:00:16-05:00


### Extracción completa

Se recorren las páginas configuradas con una pausa de cortesía entre peticiones y se
eliminan las referencias repetidas (el mismo producto puede aparecer en dos páginas si
el catálogo se reordena entre una petición y otra).

In [3]:
productos = scraper.scrape(ajustes)
tabla_cruda = transform.a_dataframe(scraper.a_filas(productos))
print(f"Productos únicos extraídos: {len(productos)}")
tabla_cruda.head(5)

Productos únicos extraídos: 40


,Nombre,Marca,SKU de referencia,Precio mínimo,Precio máximo,Moneda,Disponible,Categoría,URL,Fecha de extracción
0,Tenis Deportivo de Mujer Skechers 117863wlv,Skechers,461841,239920.0,239920.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117863wlv-2...,2026-11-26T16:00:16-05:00
1,Tenis Deportivo de Mujer Skechers 117704bbk,Skechers,461799,199920.0,199920.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117704bbk-2...,2026-11-26T16:00:16-05:00
2,Tenis Deportivo de Mujer Skechers 117679ofwt,Skechers,461787,229900.0,229900.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117679ofwt-...,2026-11-26T16:00:16-05:00
3,Tenis Deportivo de Mujer Skechers 117861ltmv,Skechers,461835,249900.0,249900.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117861ltmv-...,2026-11-26T16:00:16-05:00
4,Tenis Deportivo de Mujer Skechers 117736wht,Skechers,461817,223920.0,223920.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117736wht-2...,2026-11-26T16:00:16-05:00


## 3. Limpieza y reglas de calidad

La extracción no se publica tal cual. Antes pasa por reglas explícitas, y si alguna
falla el pipeline se detiene: es preferible un tablero viejo que un tablero con datos
rotos.

Las reglas son cuatro: precios convertidos a número, sin filas sin nombre o sin precio,
sin referencias duplicadas y una completitud mínima del 90 % de los campos. La función
`validar` devuelve la lista de problemas; una lista vacía significa que el dataset es
publicable.

In [4]:
tabla = transform.limpiar(tabla_cruda)
problemas = transform.validar(tabla)

print(f"Filas después de limpiar: {len(tabla)}")
print("Problemas de calidad:", problemas if problemas else "ninguno")
tabla.head(5)

Filas después de limpiar: 40
Problemas de calidad: ninguno


,Nombre,Marca,SKU de referencia,Precio mínimo,Precio máximo,Moneda,Disponible,Categoría,URL,Fecha de extracción
0,Tenis Deportivo de Mujer Skechers 117863wlv,Skechers,461841,239920.0,239920.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117863wlv-2...,2026-11-26T16:00:16-05:00
1,Tenis Deportivo de Mujer Skechers 117704bbk,Skechers,461799,199920.0,199920.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117704bbk-2...,2026-11-26T16:00:16-05:00
2,Tenis Deportivo de Mujer Skechers 117679ofwt,Skechers,461787,229900.0,229900.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117679ofwt-...,2026-11-26T16:00:16-05:00
3,Tenis Deportivo de Mujer Skechers 117861ltmv,Skechers,461835,249900.0,249900.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117861ltmv-...,2026-11-26T16:00:16-05:00
4,Tenis Deportivo de Mujer Skechers 117736wht,Skechers,461817,223920.0,223920.0,COP,Disponible,https://www.calzatodo.com.co/es/mujeres/tenis/...,https://www.calzatodo.com.co/tenis-117736wht-2...,2026-11-26T16:00:16-05:00


## 4. Métricas: qué dicen los datos

Las cifras salen de la tabla limpia, nunca de valores escritos a mano. Además del
conteo y los precios, se calculan dos medidas que conviene mirar juntas:

- **Promedio**: suma de los precios dividida entre el número de publicaciones. Lo
  mueven las referencias caras.
- **Mediana**: el precio que queda en el medio al ordenar de menor a mayor. No la
  mueven los extremos y representa mejor a la publicación típica.

Si la mediana es mucho menor que el promedio, hay pocas referencias caras estirando la
distribución. Eso es exactamente lo que se va a ver acá, y es una conclusión del
proyecto, no una suposición.

In [5]:
metricas = transform.resumen(tabla)

resumen = pd.DataFrame(
    [
        ("Publicaciones únicas", metricas["total_productos"]),
        ("Marcas", metricas["marcas_unicas"]),
        ("Precio mínimo", report.pesos(metricas["precio_minimo"])),
        ("Precio máximo", report.pesos(metricas["precio_maximo"])),
        ("Precio promedio", report.pesos(metricas["precio_promedio"])),
        ("Precio mediano", report.pesos(metricas["precio_mediana"])),
        ("Completitud de campos", f"{metricas['completitud_pct']:.1f} %"),
    ],
    columns=["Indicador", "Valor"],
)
resumen

,Indicador,Valor
0,Publicaciones únicas,40
1,Marcas,3
2,Precio mínimo,$ 107.920
3,Precio máximo,$ 279.920
4,Precio promedio,$ 151.168
5,Precio mediano,$ 111.920
6,Completitud de campos,100.0 %


In [6]:
por_marca = (
    tabla.groupby("Marca")
    .agg(publicaciones=("SKU de referencia", "count"),
         precio_promedio=("Precio mínimo", "mean"),
         precio_mediano=("Precio mínimo", "median"))
    .sort_values("publicaciones", ascending=False)
)
por_marca["precio_promedio"] = por_marca["precio_promedio"].map(report.pesos)
por_marca["precio_mediano"] = por_marca["precio_mediano"].map(report.pesos)
por_marca

,publicaciones,precio_promedio,precio_mediano
Marca,,,
Throwing,27,$ 111.624,$ 111.920
Skechers,8,$ 220.915,$ 223.920
New balance,5,$ 253.116,$ 249.900


In [7]:
rangos = pd.DataFrame(metricas["rangos_precio"]).rename(
    columns={"rango": "Rango de precio", "cantidad": "Publicaciones"}
)
rangos["Participación"] = (rangos["Publicaciones"] / len(tabla) * 100).round(1).astype(str) + " %"
rangos

,Rango de precio,Publicaciones,Participación
0,Menos de $150.000,27,67.5 %
1,$150.000 – $250.000,11,27.5 %
2,$250.000 – $350.000,2,5.0 %
3,$350.000 o más,0,0.0 %


## 5. Publicación: CSV, métricas y tablero

Con los datos validados se guardan tres archivos: el CSV (para reutilizarlo), las
métricas en JSON (para que el informe cite las mismas cifras) y el tablero HTML (lo que
se publica en GitHub Pages). El tablero se genera con una plantilla y toma los datos de
la tabla, así que no puede quedar desactualizado respecto al CSV.

In [8]:
transform.guardar_csv(tabla, ajustes.csv_path)
transform.guardar_metricas(metricas, ajustes.metricas_path)
ruta_tablero = report.construir_tablero(tabla, metricas, ajustes)

for ruta in (ajustes.csv_path, ajustes.metricas_path, ruta_tablero):
    print(f"{ruta.relative_to(RAIZ)} — {ruta.stat().st_size:,} bytes")

data/productos.csv — 9,290 bytes
data/metricas.json — 849 bytes
report/index.html — 34,022 bytes


El tablero queda en `report/index.html` y se puede abrir en el navegador. Muestra seis
indicadores, la distribución por marca, la distribución por rango de precio y un
explorador con búsqueda por producto, marca o referencia.

## 6. Pruebas automáticas

Las pruebas no salen a internet: usan un recorte real de la página guardado en
`tests/fixtures/` y comprueban el parser, la conversión de precios, los reintentos, la
deduplicación, las reglas de calidad, las métricas y el tablero. Una prueba aparte,
marcada `integracion`, consulta el sitio real solo si se pide con `EA3_LIVE=1`.

Este bloque ejecuta la misma orden que corre GitHub Actions:

In [9]:
import subprocess
resultado = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=RAIZ, capture_output=True, text=True,
)
print(resultado.stdout.strip())
print("Código de salida:", resultado.returncode)

...............s......                                                   [100%]
Código de salida: 0


## 7. Automatización y despliegue (GitHub Actions)

Los tres workflows del repositorio forman el ciclo de CI/CD:

1. **`ci.yml`** — en cada push o pull request: instala dependencias, corre Ruff
   (errores y estilo), ejecuta las pruebas y valida la calidad del dataset. Si algo
   falla, el check queda en rojo.
2. **`deploy.yml`** — en cada push a `main`: repite las pruebas y, solo si pasan,
   regenera el tablero y lo publica en GitHub Pages. La URL queda en el job de
   despliegue.
3. **`scrape.yml`** — todos los miércoles a las 19:00 UTC: extrae el catálogo real,
   versiona el CSV, las métricas y el tablero con un commit automático, y vuelve a
   desplegar. Se detiene cuando la fecha supera la vigencia del curso
   (`2026-12-09`).

El token de escritura lo genera GitHub en cada ejecución (`secrets.GITHUB_TOKEN`);
no hay credenciales en el repositorio. Los parámetros (`EA3_PAGES`,
`EA3_VIGENCIA_HASTA`) se leen de las variables del repositorio.

- Repositorio: https://github.com/Estefa-M/EA2_Web_Scraping
- Tablero publicado: https://estefa-m.github.io/EA2_Web_Scraping/

## 8. Resultados y conclusiones

- La categoría muestra **40 publicaciones únicas** de **3 marcas**. Throwing concentra
  27, seguida de Skechers (8) y New Balance (5).
- El precio más bajo es **$ 107.920** y el más alto **$ 279.920**. La mediana
  ($ 111.920) está muy por debajo del promedio ($ 151.168): la mayoría del catálogo es
  económico y unas pocas referencias caras suben el promedio. Por eso el tablero
  muestra las dos medidas y no solo el promedio.
- El **67,5 %** de las publicaciones cuesta menos de $150.000; solo 2 superan los
  $250.000.
- La completitud de campos es del **100 %**, y las pruebas (22) pasan sin conexión: el
  pipeline puede correr en un runner limpio y fallar solo si el sitio cambia o si los
  datos dejan de cumplir las reglas.
- El valor del trabajo no está en el número de filas, sino en que **cualquiera puede
  reproducir la cifra**: el CSV, las métricas, el tablero y el informe salen del mismo
  comando.

**Limitaciones.** La muestra son dos páginas de una categoría, no todo el catálogo; los
precios son una foto del momento; y el parser depende de que el sitio mantenga el bloque
JSON-LD. La ejecución programada termina el 9 de diciembre de 2026.

## 9. Referencias

Chacon, S. y Straub, B. (2014). *Pro Git* (2.ª ed.). Apress. https://git-scm.com/book/es/v2

GitHub. (2026). *Documentación de GitHub Actions*. https://docs.github.com/es/actions

GitHub. (2026). *Publicar sitios con GitHub Pages*. https://docs.github.com/es/pages

Richardson, L. (2024). *Beautiful Soup Documentation*. https://www.crummy.com/software/BeautifulSoup/bs4/doc/

The pandas development team. (2026). *pandas documentation*. https://pandas.pydata.org/docs/